# Simulating randomized benchmarking with Quax

This notebook demonstrates how we can construct and simulate randomized benchmarking sequences using quax.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp
import quax as qx

# enable 64 bit
jax.config.update("jax_enable_x64", True)

## Parameters

In [ ]:
num_randomizations = 30
depths = [2, 4, 8, 16, 32, 64, 128, 256]
seed = 485
t1 = 30e-6
t2 = 35e-6
gate_time = 50e-9

Generate the noisy superoperators

In [ ]:
t1s = jnp.array([t1])
t2s = jnp.array([t2])
gate_time = gate_time

### Generate the RB sequences

First, let's generate the sets of random 1Q Clifford gates and the inversions

In [ ]:
key = jax.random.key(seed)

num_cliffords = qx.ensembles.CLIFFORDS_1Q.ensemble_size[0]


def invert_sequence(random_unitaries: qx.Unitary) -> qx.Unitary:
    """
    Given a sequence of 1Q unitaries, compute the overall unitary and invert it.

    :param random_unitaries: Ensemble of random sequences dimension (num_randomizations, depth, 2, 2).
    :return: (num_randomizations, depth + 1, 2, 2) ensemble which accumuluate to the identity.
    """

    def thunk(carry, u):
        return qx.Unitary(data=u, num_ensemble_dims=1) @ carry, None

    # compute the reduced process to find the inversion clifford
    identity = qx.Unitary(
        jnp.broadcast_to(qx.gates.I.data, (num_randomizations,) + qx.gates.I.data.shape), num_ensemble_dims=1
    )
    reduced_process, _ = jax.lax.scan(thunk, identity, random_unitaries.data)
    inversion = reduced_process.h

    # append the inversion unitary
    random_unitaries = qx.Unitary(
        data=jnp.concatenate([random_unitaries.data, inversion.data[jnp.newaxis, :, :, :]], axis=0),
        num_ensemble_dims=2,
    )

    return random_unitaries


# generate a set of Cliffords (num_randomizations x len(depths))
cliffords = []
for depth in depths:
    subkey, key = jax.random.split(key)
    rand_ints = jax.random.randint(
        subkey,
        shape=(depth - 1, num_randomizations),
        minval=0,
        maxval=num_cliffords,
    )
    # sample the random cliffords
    random_cliffords = qx.Unitary(data=qx.ensembles.CLIFFORDS_1Q.data[rand_ints], num_ensemble_dims=2)

    # invert the sequence and append
    random_cliffords = invert_sequence(random_cliffords)

    # collect the full sequence
    cliffords.append(random_cliffords)

Compute the final states

In [ ]:
initial_state = qx.zero_state_vector(1, (num_randomizations,))


def thunk(state, clifford_data):
    # jax.lax.scan slices out the leading dimension, so we need to
    # reconstruct the Unitary with the correct num_ensemble_dims
    clifford = qx.Unitary(data=clifford_data, num_ensemble_dims=1)
    new_state = clifford @ state
    return new_state, None


for i, depth_cliffords in enumerate(cliffords):
    final_state, _ = jax.lax.scan(thunk, initial_state, depth_cliffords.data)
    zero_probs = qx.bitstring_probability(final_state, jnp.array([0]))
    print(f"Probability of measuring |0> at the end of sequences for depth {depths[i]}: {zero_probs.mean():.4f}")

### Decompose to ZXZXZ

The sequence of gates which are `(depth, num_randomizations, 2, 2)` can be decomposed to native unitary operations. This yields a unitary array of dimension `(depth, num_randomizations, 5, 2, 2)`.

We perform the decomposition and see that the final states are still the 0 bitstring.

In [ ]:
zxzxz_cliffords = []


def to_zxzxz(random_sequence: qx.Unitary) -> qx.Unitary:
    """
    Convert a sequence of 1Q Cliffords to a sequence of ZXZXZ gates.

    :param random_sequence: (depth, num_randomizations, 2, 2) ensemble of random sequences.
    :return: (depth*5, num_randomizations, 2, 2) ensemble of ZXZXZ angles.
    """
    # angles is (depth, num_randomizations, 3) array of angles for each gate in the sequence
    angles = qx.to_zxzxz_angles(random_sequence)
    # build the seqeucence of RZ RX RZ RX RZ unitaries
    return angles


# for i, depth_cliffords in enumerate(cliffords):
#     zxzxz_cliffords.append(qx.to_zxzxz_angles(depth_cliffords))

qx.gates.RZ(qx.to_zxzxz_angles(cliffords[1]))

### Adding noise

We wish to add noise into the equation. For this, we'll need to promote our unitaries to superoperators.

In [ ]:
choi = qx.thermal_relaxation_choi(t1s, t2s, gate_time)
initial_state = qx.zero_state_matrix(1, (num_randomizations,))


def thunk(state, clifford_data):
    # jax.lax.scan slices out the leading dimension, so we need to
    # reconstruct the Unitary with the correct num_ensemble_dims
    clifford = qx.SuperOp(data=clifford_data, num_ensemble_dims=1)
    new_state = clifford @ state
    return new_state, new_state


# Pass the raw data array to scan, rebuild Unitary inside the function
final_states = []
for depth_cliffords in noisy_cliffords:
    final_state, _ = jax.lax.scan(thunk, initial_state, depth_cliffords.data)
    final_states.append(final_state)

Next, we'll need to generate a channel. We'll create a simple thermal relaxation channel here using the Lindbladian.

We can once again compute the probabiltiy of measuring 0. While with the unitary operators, the probability of measuring 0 was always 1.0, with the noisy gates, we'll find that the probability decreases with depth.

We can make a simple plot of the decay and see that it is as expected.

## Adding the third state

Ideally, superconducting qubits are two-state systems, but in reality they have many excited states. Sometimes, a qubit can become excited to the $|2\rangle$ state. We can model this by promoting our superoperators from qubit superoperators to qutrit superoperators. We'll do this, and set up a simple channel which has some leakage probability as well as a seepage probability.

We can repeat the experiment, but this time we'll observe the probability of measuring 0, 1 and 2.

### Determining the sensitivity

Jax enables the simple determination of gradients. This is useful in many scenarios, but here we will use it to determine the sensitivity of the bitstring probability to the noise parameters. We'll compute the gradient of the output bitstring with respect to each parameter, learning which how much each sort of error affects the overall fidelity.

### Fitting a model to data

Another thing we can use the gradenits to do is fit models. Here, we will produce a dataset using some random noise parameters and sample outcome, as if we were doing a real experiment.

We can then fit our noise model the sampled probabilities and see if we recover the result.

## Interleaved Randomized Benchmarking (IRB)

Next, we'll do the same experiment with interleaved randomized benchmarking. This case is slightly more complicated as we'll be using both 1Q and 2Q gates.